In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

BASE_PATH = "/kaggle/input/datasets/syxlicheng/heartdatabase/EchoNet-Dynamic"

trace_file = BASE_PATH + "/VolumeTracings.csv"
filelist_file = BASE_PATH + "/FileList.csv"

trace_df = pd.read_csv(trace_file)
file_df = pd.read_csv(filelist_file)

In [ ]:
print("=== VolumeTracings.csv ===")
print("Shape:", trace_df.shape)

print("\nColumns:")
print(trace_df.columns)

print("\nSample data:")
print(trace_df.head())

In [ ]:
print("\nUnique videos:", trace_df['FileName'].nunique())
print("Sample video names:", trace_df['FileName'].unique()[:5])

print("\nFrame values:")
print(trace_df['Frame'].unique()[:10])

In [ ]:
sample_video = trace_df['FileName'].iloc[0]

video_data = trace_df[trace_df['FileName'] == sample_video]

print("Video:", sample_video)
print("Frames available:", video_data['Frame'].unique())
print("Total rows for this video:", len(video_data))

In [ ]:
print("\n=== FileList.csv ===")
print("Shape:", file_df.shape)

print("\nColumns:")
print(file_df.columns)

print("\nSample data:")
print(file_df.head())

In [ ]:
print("\nEF values stats:")
print(file_df['EF'].describe())

In [ ]:
print("\nDataset split:")
print(file_df['Split'].value_counts())

In [ ]:
file_df['FileName'] = file_df['FileName'] + ".avi"

In [ ]:
common = set(trace_df['FileName']).intersection(set(file_df['FileName']))
print("Common videos:", len(common))

In [ ]:
import shutil
import os

OUTPUT_IMG = "/kaggle/working/images"
OUTPUT_MASK = "/kaggle/working/masks"

# remove old folders completely
if os.path.exists(OUTPUT_IMG):
    shutil.rmtree(OUTPUT_IMG)

if os.path.exists(OUTPUT_MASK):
    shutil.rmtree(OUTPUT_MASK)

# recreate clean folders
os.makedirs(OUTPUT_IMG, exist_ok=True)
os.makedirs(OUTPUT_MASK, exist_ok=True)

print("✅ Output folders cleaned")

In [ ]:
import cv2
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from scipy.interpolate import interp1d

# =========================
# PATHS
# =========================
BASE_PATH = "/kaggle/input/datasets/syxlicheng/heartdatabase/EchoNet-Dynamic"
VIDEO_DIR = BASE_PATH + "/Videos"
TRACE_FILE = BASE_PATH + "/VolumeTracings.csv"

OUTPUT_IMG = "/kaggle/working/images"
OUTPUT_MASK = "/kaggle/working/masks"

# =========================
# CLEAN OUTPUT (IMPORTANT)
# =========================
import shutil

if os.path.exists(OUTPUT_IMG):
    shutil.rmtree(OUTPUT_IMG)
if os.path.exists(OUTPUT_MASK):
    shutil.rmtree(OUTPUT_MASK)

os.makedirs(OUTPUT_IMG, exist_ok=True)
os.makedirs(OUTPUT_MASK, exist_ok=True)

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(TRACE_FILE)

grouped = df.groupby(["FileName", "Frame"])

# =========================
# SMOOTHING FUNCTION 🔥
# =========================
def smooth_curve(points, num=120):
    points = points[np.argsort(points[:,1])]  # sort by Y

    y = points[:,1]
    x = points[:,0]

    # remove duplicate y (important for interpolation)
    y_unique, idx = np.unique(y, return_index=True)
    x_unique = x[idx]

    if len(y_unique) < 2:
        return points  # fallback

    f = interp1d(y_unique, x_unique, kind='linear', fill_value="extrapolate")

    y_new = np.linspace(y_unique.min(), y_unique.max(), num)
    x_new = f(y_new)

    return np.stack([x_new, y_new], axis=1)

count = 0

# =========================
# MAIN LOOP
# =========================
for (video_name, frame_id), group in tqdm(grouped):

    video_path = os.path.join(VIDEO_DIR, video_name)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        continue

    h, w = frame.shape[:2]

    # =========================
    # EXTRACT CONTOURS
    # =========================
    pts1 = group[['X1', 'Y1']].values
    pts2 = group[['X2', 'Y2']].values

    # =========================
    # SMOOTH BOTH SIDES 🔥
    # =========================
    pts1_smooth = smooth_curve(pts1)
    pts2_smooth = smooth_curve(pts2)

    # reverse right side
    pts2_smooth = pts2_smooth[::-1]

    # combine to form closed contour
    pts = np.vstack((pts1_smooth, pts2_smooth))
    pts = pts.astype(np.int32)

    # =========================
    # CREATE MASK
    # =========================
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 255)

    # optional light smoothing (not aggressive)
    mask = cv2.GaussianBlur(mask, (3,3), 0)

    # =========================
    # SAVE
    # =========================
    name = f"{video_name}_frame{frame_id}.png"

    cv2.imwrite(os.path.join(OUTPUT_IMG, name), frame)
    cv2.imwrite(os.path.join(OUTPUT_MASK, name), mask)

    count += 1

print("✅ Total samples created:", count)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

sample = os.listdir(OUTPUT_IMG)[10]

img = cv2.imread(os.path.join(OUTPUT_IMG, sample))
mask = cv2.imread(os.path.join(OUTPUT_MASK, sample), 0)

# overlay
overlay = img.copy()
overlay[mask > 0] = [0, 255, 0]

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Image")
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(1,3,2)
plt.title("Mask")
plt.imshow(mask, cmap='gray')
plt.axis('off')

plt.subplot(1,3,3)
plt.title("Overlay")
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.show()

In [ ]:
import os

print("Images:", len(os.listdir("/kaggle/working/images")))
print("Masks :", len(os.listdir("/kaggle/working/masks")))

In [ ]:
import os
import shutil

combined_path = "/kaggle/working/dataset"

os.makedirs(combined_path, exist_ok=True)

shutil.copytree("/kaggle/working/images", combined_path + "/images")
shutil.copytree("/kaggle/working/masks", combined_path + "/masks")

shutil.make_archive('/kaggle/working/echonet_dataset', 'zip', combined_path)

print("✅ Combined dataset ready")

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/echonet_dataset', 'zip', '/kaggle/working')

print("✅ Zip created")